# Hyperparameter Tuning — Optuna sobre LightGBM
60 trials de búsqueda bayesiana (TPE) con 3-fold CV.
Compara el modelo tuneado contra el baseline de `train_06.py`.

In [1]:
import sys
from pathlib import Path
ROOT = Path('..').resolve()
sys.path.append(str(ROOT))

from src.models.tune_07 import main, plot_tuning_history, N_TRIALS
from src.models.train_06 import load_data, evaluate_on_test, plot_roc_pr, MODELS_DIR

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import optuna
from sklearn.model_selection import train_test_split
sns.set_theme(style='whitegrid')
%matplotlib inline
RANDOM_STATE = 42

/Users/josefrodriguez/repos_publicar/Churn_project/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Ejecutar tuning

In [2]:
tuned_model, study = main()

── Cargando features ──
Dataset: 992,931 filas × 30 features | churn rate: 6.39%

── Optimización Optuna (60 trials × 3-fold CV) ──


Best trial: 0. Best value: 0.985161:   2%|▏         | 1/60 [02:16<2:14:09, 136.43s/it]


[W 2026-04-25 00:34:13,142] Trial 1 failed with parameters: {'n_estimators': 767, 'learning_rate': 0.010485387725194618, 'num_leaves': 249, 'max_depth': 9, 'min_child_samples': 58, 'subsample': 0.6727299868828402, 'colsample_bytree': 0.6733618039413735, 'reg_alpha': 0.003320559103751959, 'reg_lambda': 0.042051564509138696} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/Users/josefrodriguez/repos_publicar/Churn_project/.venv/lib/python3.11/site-packages/optuna/study/_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "/Users/josefrodriguez/repos_publicar/Churn_project/src/models/tune_07.py", line 61, in objective
    model.fit(X_train[train_idx], y_train[train_idx])
  File "/Users/josefrodriguez/repos_publicar/Churn_project/.venv/lib/python3.11/site-packages/lightgbm/sklearn.py", line 1560, in fit
    super().fit(
  File "/Users/josefrodriguez/repos_publicar/Churn_project/.ven

KeyboardInterrupt: 

## 2. Historial de optimización

In [ ]:
plot_tuning_history(study)

## 3. Mejores parámetros

In [ ]:
trials = study.trials_dataframe().sort_values('value', ascending=False)
print(f'Mejor AUC CV: {study.best_value:.5f}')
print('\nMejores parámetros:')
for k, v in study.best_params.items():
    print(f'  {k:25s}: {v}')
trials.head(10)

## 4. Importancia de parámetros

In [ ]:
importances = optuna.importance.get_param_importances(study)
fig, ax = plt.subplots(figsize=(9, 4))
ax.barh(list(importances.keys())[::-1], list(importances.values())[::-1],
        color='#2196F3', edgecolor='white')
ax.set_title('Importancia de hiperparámetros (Optuna FAnova)')
ax.set_xlabel('Importancia relativa')
plt.tight_layout()
plt.show()

## 5. Cargar modelos y comparar en test

In [ ]:
X, y = load_data()
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)

baseline = joblib.load(MODELS_DIR / 'best_model.joblib')['model']
tuned    = joblib.load(MODELS_DIR / 'lgbm_tuned.joblib')['model']

results = [
    evaluate_on_test('LightGBM baseline', baseline, X_test, y_test),
    evaluate_on_test('LightGBM tuned',    tuned,    X_test, y_test),
]
plot_roc_pr(results, y_test)